# SmolVLA Fine-Tuning on Berkeley UR5 Dataset

This notebook walks through the complete pipeline of fine-tuning HuggingFace's **SmolVLA** Vision-Language-Action model on the **Berkeley AutoLab UR5** dataset using the **LeRobot** framework.

**What you'll do:**
1. Install dependencies
2. Load and explore the Berkeley UR5 dataset
3. Visualize sample trajectories
4. Configure and launch SmolVLA fine-tuning
5. Monitor training
6. Evaluate the trained policy
7. Export the model for deployment

---
## 1. Environment Setup

In [ ]:
# Install LeRobot with SmolVLA support
# Run this once, then restart the kernel
!pip install -q "lerobot[smolvla]" wandb matplotlib

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
import os

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# (Optional) Login to HuggingFace Hub
from huggingface_hub import login
login()  # Paste your token when prompted

---
## 2. Load & Explore the Berkeley UR5 Dataset

In [ ]:
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

# Load the dataset (downloads from HuggingFace Hub on first run)
DATASET_REPO_ID = "lerobot/berkeley_autolab_ur5"
dataset = LeRobotDataset(DATASET_REPO_ID)

print("=" * 60)
print("BERKELEY UR5 DATASET INFO")
print("=" * 60)
print(f"Total episodes:  {dataset.meta.total_episodes}")
print(f"Total frames:    {dataset.meta.total_frames}")
print(f"Total videos:    {dataset.meta.total_videos}")
print(f"FPS:             {dataset.meta.fps}")
print(f"Total tasks:     {dataset.meta.total_tasks}")
print(f"\nFeatures:")
for name, info in dataset.meta.features.items():
    if isinstance(info, dict):
        shape = info.get('shape', 'N/A')
        dtype = info.get('dtype', 'N/A')
        print(f"  {name:45s} | shape={shape} | dtype={dtype}")

In [ ]:
# Inspect a single sample
sample = dataset[0]
print("\nSample keys and shapes:")
print("-" * 60)
for key, value in sample.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key:45s} | {str(value.shape):20s} | {value.dtype}")
    elif isinstance(value, str):
        print(f"  {key:45s} | str: '{value[:60]}...'" if len(str(value)) > 60 else f"  {key:45s} | str: '{value}'")
    else:
        print(f"  {key:45s} | {type(value).__name__}: {value}")

---
## 3. Visualize Sample Trajectories

In [ ]:
def visualize_episode(dataset, episode_idx=0, num_frames=8):
    """
    Visualize frames from a single episode showing all camera views.
    """
    # Get indices for this episode
    ep_start = None
    ep_end = None
    for i in range(len(dataset)):
        sample = dataset[i]
        if sample.get('episode_index', sample.get('episode_index', -1)) == episode_idx:
            if ep_start is None:
                ep_start = i
            ep_end = i
        elif ep_start is not None:
            break
    
    if ep_start is None:
        print(f"Episode {episode_idx} not found")
        return
    
    ep_length = ep_end - ep_start + 1
    step = max(1, ep_length // num_frames)
    frame_indices = range(ep_start, ep_end + 1, step)[:num_frames]
    
    # Determine available camera keys
    camera_keys = [k for k in dataset[ep_start].keys() if 'image' in k and isinstance(dataset[ep_start][k], torch.Tensor)]
    
    fig, axes = plt.subplots(len(camera_keys), num_frames, 
                              figsize=(3 * num_frames, 3 * len(camera_keys)))
    if len(camera_keys) == 1:
        axes = axes.reshape(1, -1)
    
    for col, idx in enumerate(frame_indices):
        sample = dataset[idx]
        for row, cam_key in enumerate(camera_keys):
            img = sample[cam_key]
            if img.dim() == 3 and img.shape[0] in [1, 3]:
                img = img.permute(1, 2, 0)
            img_np = img.numpy()
            if img_np.max() <= 1.0:
                img_np = (img_np * 255).astype(np.uint8)
            axes[row, col].imshow(img_np)
            axes[row, col].axis('off')
            if col == 0:
                short_name = cam_key.split('.')[-1]
                axes[row, col].set_ylabel(short_name, fontsize=10, rotation=0, labelpad=80)
            if row == 0:
                axes[row, col].set_title(f'Frame {idx - ep_start}', fontsize=9)
    
    # Get task instruction
    task = dataset[ep_start].get('language_instruction', 'N/A')
    fig.suptitle(f'Episode {episode_idx} — "{task}"', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Visualize the first episode
visualize_episode(dataset, episode_idx=0, num_frames=6)

In [ ]:
def plot_episode_actions(dataset, episode_idx=0):
    """
    Plot the action trajectory for a given episode.
    """
    actions = []
    states = []
    found = False
    
    for i in range(len(dataset)):
        sample = dataset[i]
        ep_idx = sample.get('episode_index', -1)
        if isinstance(ep_idx, torch.Tensor):
            ep_idx = ep_idx.item()
        if ep_idx == episode_idx:
            found = True
            actions.append(sample['action'].numpy())
            if 'observation.state' in sample:
                states.append(sample['observation.state'].numpy())
        elif found:
            break
    
    actions = np.array(actions)
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Plot actions
    action_labels = [f'action_{i}' for i in range(actions.shape[1])]
    for i in range(actions.shape[1]):
        axes[0].plot(actions[:, i], label=action_labels[i], alpha=0.7)
    axes[0].set_title(f'Episode {episode_idx} — Actions Over Time', fontsize=13)
    axes[0].set_xlabel('Timestep')
    axes[0].set_ylabel('Action Value')
    axes[0].legend(loc='upper right', fontsize=8)
    axes[0].grid(True, alpha=0.3)
    
    # Plot states
    if states:
        states = np.array(states)
        for i in range(min(states.shape[1], 8)):
            axes[1].plot(states[:, i], label=f'state_{i}', alpha=0.7)
        axes[1].set_title(f'Episode {episode_idx} — Robot State Over Time', fontsize=13)
        axes[1].set_xlabel('Timestep')
        axes[1].set_ylabel('State Value')
        axes[1].legend(loc='upper right', fontsize=8)
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    print(f"Action dimension: {actions.shape[1]}")
    print(f"Action range: [{actions.min():.4f}, {actions.max():.4f}]")

plot_episode_actions(dataset, episode_idx=0)

---
## 4. Configure Training

We'll fine-tune SmolVLA using the LeRobot training infrastructure. Key hyperparameters are explained below.

In [ ]:
# ──── Training Configuration ────

TRAIN_CONFIG = {
    # Model
    "policy_type": "smolvla",
    "pretrained_path": "lerobot/smolvla_base",  # base pre-trained model
    
    # Dataset
    "dataset_repo_id": "lerobot/berkeley_autolab_ur5",
    
    # Training hyperparameters
    "batch_size": 32,        # Reduce to 8 or 16 if OOM
    "learning_rate": 1e-4,   # Conservative for fine-tuning
    "training_steps": 20000, # ~205 epochs over 1000 episodes
    "num_workers": 4,        # Data loading workers
    
    # Output
    "output_dir": "outputs/train/smolvla_berkeley_ur5",
    "job_name": "smolvla_berkeley_ur5",
    
    # Logging
    "wandb_enable": False,   # Set True if you have W&B configured
    
    # Hardware
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

print("Training Configuration:")
print("=" * 50)
for k, v in TRAIN_CONFIG.items():
    print(f"  {k:25s}: {v}")

### 4.1 Understanding the Hyperparameters

| Parameter | Value | Rationale |
|---|---|---|
| `batch_size` | 32 | Good balance of gradient stability and VRAM usage |
| `learning_rate` | 1e-4 | Standard for VLA fine-tuning; prevents catastrophic forgetting |
| `training_steps` | 20,000 | ~200 passes over 97.9K frames with batch 32 |
| `pretrained_path` | smolvla_base | Uses HF pre-trained weights as initialization |

---
## 5. Launch Training

We run training via the LeRobot CLI. This cell builds the command and executes it.

In [ ]:
# Build the training command
train_cmd = f"""
python -m lerobot.scripts.train \\
    --policy.type={TRAIN_CONFIG['policy_type']} \\
    --policy.pretrained_path={TRAIN_CONFIG['pretrained_path']} \\
    --dataset.repo_id={TRAIN_CONFIG['dataset_repo_id']} \\
    --batch_size={TRAIN_CONFIG['batch_size']} \\
    --steps={TRAIN_CONFIG['training_steps']} \\
    --output_dir={TRAIN_CONFIG['output_dir']} \\
    --job_name={TRAIN_CONFIG['job_name']} \\
    --policy.device={TRAIN_CONFIG['device']} \\
    --policy.optimizer_lr={TRAIN_CONFIG['learning_rate']} \\
    --num_workers={TRAIN_CONFIG['num_workers']} \\
    --wandb.enable={str(TRAIN_CONFIG['wandb_enable']).lower()}
""".strip()

print("Training command:")
print("=" * 70)
print(train_cmd)
print("\n" + "=" * 70)
print("\n⚠️  Run the cell below to start training.")
print("    Expected time: 2-6 hours on an RTX 4090 / A100.")
print("    Reduce batch_size if you encounter CUDA OOM errors.")

In [ ]:
# ╔═══════════════════════════════════════════════╗
# ║  🚀 START TRAINING — Execute this cell         ║
# ╚═══════════════════════════════════════════════╝

!{train_cmd}

---
## 6. Monitor & Analyze Training

After training completes (or while it runs), you can analyze the results.

In [ ]:
# Check what was saved
output_dir = Path(TRAIN_CONFIG['output_dir'])

if output_dir.exists():
    print("Output directory contents:")
    for item in sorted(output_dir.rglob('*')):
        if item.is_file():
            size_mb = item.stat().st_size / 1e6
            rel = item.relative_to(output_dir)
            print(f"  {str(rel):50s} ({size_mb:.1f} MB)")
else:
    print(f"❌ Output directory not found: {output_dir}")
    print("   Make sure training has completed or is running.")

In [ ]:
# Launch TensorBoard (uncomment to use inline in notebook)
# %load_ext tensorboard
# %tensorboard --logdir {TRAIN_CONFIG['output_dir']}

---
## 7. Load & Evaluate the Trained Policy

In [ ]:
from lerobot.common.policies.smolvla.modeling_smolvla import SmolVLAPolicy

# Load the last checkpoint
checkpoint_path = Path(TRAIN_CONFIG['output_dir']) / 'checkpoints' / 'last'
print(f"Loading policy from: {checkpoint_path}")

policy = SmolVLAPolicy.from_pretrained(str(checkpoint_path))
policy.eval()

device = torch.device(TRAIN_CONFIG['device'])
policy = policy.to(device)
print(f"Policy loaded on {device}")
print(f"Total parameters: {sum(p.numel() for p in policy.parameters()) / 1e6:.1f}M")

In [ ]:
def run_inference_on_sample(policy, dataset, sample_idx=0, device='cuda'):
    """
    Run the policy on a single dataset sample and visualize predictions.
    """
    sample = dataset[sample_idx]
    
    # Build batch (add batch dimension)
    batch = {}
    for key, value in sample.items():
        if isinstance(value, torch.Tensor):
            batch[key] = value.unsqueeze(0).to(device)
        elif isinstance(value, str):
            batch[key] = [value]
        else:
            batch[key] = value
    
    # Run inference
    with torch.no_grad():
        action_chunk = policy.select_action(batch)
    
    predicted_actions = action_chunk.cpu().numpy()
    gt_action = sample['action'].numpy()
    
    print(f"\n{'='*60}")
    print(f"Sample {sample_idx} — Inference Results")
    print(f"{'='*60}")
    print(f"Action chunk shape: {predicted_actions.shape}")
    print(f"\nGround-truth action (this frame): {gt_action}")
    print(f"Predicted action (first step):     {predicted_actions[0]}")
    print(f"\nL2 error (first step): {np.linalg.norm(predicted_actions[0] - gt_action):.6f}")
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Camera views
    cam_keys = [k for k in sample.keys() if 'image' in k and isinstance(sample[k], torch.Tensor)]
    for i, cam_key in enumerate(cam_keys[:2]):
        img = sample[cam_key]
        if img.dim() == 3 and img.shape[0] in [1, 3]:
            img = img.permute(1, 2, 0)
        img_np = img.numpy()
        if img_np.max() <= 1.0:
            img_np = (img_np * 255).astype(np.uint8)
        axes[i].imshow(img_np)
        axes[i].set_title(cam_key.split('.')[-1], fontsize=12)
        axes[i].axis('off')
    
    # Action comparison
    x = np.arange(len(gt_action))
    width = 0.35
    axes[2].bar(x - width/2, gt_action, width, label='Ground Truth', color='steelblue', alpha=0.8)
    axes[2].bar(x + width/2, predicted_actions[0][:len(gt_action)], width, label='Predicted', color='coral', alpha=0.8)
    axes[2].set_xlabel('Action Dimension')
    axes[2].set_ylabel('Value')
    axes[2].set_title('GT vs Predicted Action', fontsize=12)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    task = sample.get('language_instruction', 'N/A')
    fig.suptitle(f'Inference — "{task}"', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    return predicted_actions

# Run inference on a few samples
for idx in [0, 100, 500]:
    run_inference_on_sample(policy, dataset, sample_idx=idx, device=str(device))

In [ ]:
def evaluate_on_episodes(policy, dataset, episode_indices=[0, 1, 2], device='cuda'):
    """
    Compute average action prediction error across full episodes.
    """
    results = []
    
    for ep_idx in episode_indices:
        errors = []
        found = False
        
        for i in range(len(dataset)):
            sample = dataset[i]
            curr_ep = sample.get('episode_index', -1)
            if isinstance(curr_ep, torch.Tensor):
                curr_ep = curr_ep.item()
            
            if curr_ep == ep_idx:
                found = True
                batch = {}
                for key, value in sample.items():
                    if isinstance(value, torch.Tensor):
                        batch[key] = value.unsqueeze(0).to(device)
                    elif isinstance(value, str):
                        batch[key] = [value]
                    else:
                        batch[key] = value
                
                with torch.no_grad():
                    pred = policy.select_action(batch)
                
                gt = sample['action'].numpy()
                pred_first = pred[0].cpu().numpy()[:len(gt)]
                error = np.linalg.norm(pred_first - gt)
                errors.append(error)
            elif found:
                break
        
        if errors:
            avg_err = np.mean(errors)
            results.append({'episode': ep_idx, 'mean_l2_error': avg_err, 'num_frames': len(errors)})
            print(f"Episode {ep_idx:4d} | Mean L2 Error: {avg_err:.6f} | Frames: {len(errors)}")
    
    return results

print("Evaluating on first 5 episodes...")
print("=" * 60)
eval_results = evaluate_on_episodes(policy, dataset, episode_indices=list(range(5)), device=str(device))

---
## 8. Export the Model

Save and optionally push your fine-tuned model to the HuggingFace Hub.

In [ ]:
# Save locally
export_dir = Path("smolvla_ur5_finetuned")
export_dir.mkdir(exist_ok=True)

policy.save_pretrained(str(export_dir))
print(f"✅ Policy saved to: {export_dir.absolute()}")
print(f"\nContents:")
for f in sorted(export_dir.rglob('*')):
    if f.is_file():
        print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# (Optional) Push to HuggingFace Hub
# Uncomment and set your username below:

# HF_USERNAME = "your-username"
# REPO_NAME = "smolvla_berkeley_ur5"
# policy.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")
# print(f"✅ Pushed to: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

---
## 9. Deployment Preview

To deploy this policy on your UR5 (simulated in Isaac Sim via ROS2), see the full deployment guide in **`book.md` Section 8**.

The deployment pipeline uses a ROS2 node (`smolvla_policy_node.py`) that:
1. Subscribes to `/joint_states` and camera image topics from Isaac Sim
2. Packs observations into the format expected by SmolVLA
3. Runs `policy.select_action(obs)` to get an action chunk
4. Publishes joint commands back to Isaac Sim's controller

```
Isaac Sim (UR5)  ──ROS2──▶  Policy Node  ──ROS2──▶  Isaac Sim (UR5)
  (observations)             (SmolVLA)              (joint commands)
```

Quick launch:
```bash
# Terminal 1: Isaac Sim with UR5 + ROS2 bridge
# Terminal 2:
source /opt/ros/humble/setup.bash
conda activate lerobot
python smolvla_policy_node.py \
    --ros-args \
    -p model_path:=smolvla_ur5_finetuned \
    -p language_instruction:="Pick up the blue cup and put it into the brown cup."
```

---

**🎉 Done!** You've successfully:
1. Explored the Berkeley UR5 dataset
2. Fine-tuned SmolVLA on UR5 manipulation tasks
3. Evaluated the trained policy
4. Exported the model for deployment

Next steps:
- Deploy on Isaac Sim (see `book.md` Section 8)
- Experiment with hyperparameters (learning rate, training steps)
- Collect your own UR5 data and fine-tune further